# Lab 8　LaLonde：當橋不存在

職業訓練能讓收入增加多少？同一批受訓者，配上兩種對照組：一種是隨機分派沒被選上的人（實驗），一種是從一般族群 PSID 抓來的人（觀察）。實驗給了一個基準答案；觀察資料上，所有估計器會怎樣？這是 Lab 4 第 06 格 positivity 旋鈕轉到底的真實版本。

> **怎麼跑：** 這本筆記本用 **R** 執行（Colab 會自動選 R runtime；如果右上角不是 R，到「執行階段 → 變更執行階段類型」選 R）。從上往下依序執行：第一格下載資料、第二格載入工具箱，之後每一格都可以改了再跑。想一次跑完：「執行階段 → 全部執行」。
>
> 網頁版（不用登入、直接在瀏覽器跑）：[https://kiwi710680.github.io/Pokemon-CP/08_lalonde.html](https://kiwi710680.github.io/Pokemon-CP/08_lalonde.html)

In [ ]:
# Setup: download the data used by this lab (from https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/) and set plot size
BASE <- "https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/"
files <- c("data/lalonde/nswre74_treated.txt", "data/lalonde/nswre74_control.txt", "data/lalonde/psid_controls.txt")
for (f in files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  for (try in 1:3) {                                   # retry: a single network hiccup should not stop the lab
    ok <- tryCatch(download.file(paste0(BASE, f), f, quiet = TRUE, mode = "wb") == 0, error = function(e) FALSE)
    if (ok) break else Sys.sleep(2)
  }
  if (!ok) stop("could not download ", f, " - check the internet connection and run this cell again")
}
for (p in c("randomForest")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 100, width = 110, digits = 4)
cat(if (length(files)) paste("ready:", length(files), "data file(s)") else "ready: this lab simulates its own data", "\n")

## 工具箱

這一頁用到的 R 函式都在下面這一格：估計器、資料讀取、畫圖小工具。先執行它；想知道某個函式做什麼，回來這裡找。

In [ ]:
# ---- estimators: one set of nuisance predictions -> different answers ----
# Notation (same as the slides): O = (X, A, Y)
#   Q(a, X) = E[Y | A = a, X]            outcome regression   -> nu$Q1, nu$Q0
#   g(X)    = P(A = 1 | X)               propensity score     -> nu$g
#   H(A, X) = A / g - (1 - A) / (1 - g)  clever covariate
#   D(O; g, Q, psi) = H (Y - Q_A) + Q1 - Q0 - psi   efficient influence function of the ATE
expit <- function(x) 1 / (1 + exp(-x))
bound <- function(p, lo = 0.025, hi = 0.975) pmin(pmax(p, lo), hi)

est_plugin <- function(nu) mean(nu$Q1 - nu$Q0)               # g-computation: predict twice, subtract, average

est_ipw_ht <- function(dat, nu)                               # Horvitz-Thompson IPTW: divide by n
  mean(dat$A * dat$Y / nu$g - (1 - dat$A) * dat$Y / (1 - nu$g))

est_ipw <- function(dat, nu) {                                # Hajek (stabilized) IPTW: divide by the sum of weights
  w1 <- dat$A / nu$g; w0 <- (1 - dat$A) / (1 - nu$g)
  sum(w1 * dat$Y) / sum(w1) - sum(w0 * dat$Y) / sum(w0)
}

eif <- function(dat, nu, psi) {
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  QA <- ifelse(dat$A == 1, nu$Q1, nu$Q0)
  H * (dat$Y - QA) + nu$Q1 - nu$Q0 - psi
}

wald <- function(psi, D) {                                    # SE from the EIF: sqrt(mean(D^2) / n)
  se <- sqrt(mean(D^2) / length(D))
  c(est = psi, se = se, lo = psi - 1.96 * se, hi = psi + 1.96 * se)
}

est_aipw <- function(dat, nu) {                               # one-step: psi_plug + P_n D  (solves P_n D = 0)
  psi <- mean(eif(dat, nu, 0))
  wald(psi, eif(dat, nu, psi))
}

est_tmle <- function(dat, nu, ybounds = range(dat$Y)) {       # targeting: move Q-hat along H until P_n D = 0
  lo <- ybounds[1]; hi <- ybounds[2]
  sc <- function(y) pmin(pmax((y - lo) / (hi - lo), 1e-6), 1 - 1e-6)
  Ys <- (dat$Y - lo) / (hi - lo)
  Q1s <- sc(nu$Q1); Q0s <- sc(nu$Q0); QAs <- ifelse(dat$A == 1, Q1s, Q0s)
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  fl <- suppressWarnings(glm(Ys ~ -1 + H, offset = qlogis(QAs), family = quasibinomial()))
  eps <- unname(coef(fl))
  Q1t <- expit(qlogis(Q1s) + eps / nu$g)
  Q0t <- expit(qlogis(Q0s) - eps / (1 - nu$g))
  nu_t <- data.frame(Q1 = lo + (hi - lo) * Q1t, Q0 = lo + (hi - lo) * Q0t, g = nu$g)
  psi <- mean(nu_t$Q1 - nu_t$Q0)
  c(wald(psi, eif(dat, nu_t, psi)), eps = eps)
}

weight_summary <- function(dat, nu) {                         # largest weight and effective sample size (ESS)
  w <- ifelse(dat$A == 1, 1 / nu$g, 1 / (1 - nu$g))
  c(max_weight = max(w),
    ESS_treated = sum(w[dat$A == 1])^2 / sum(w[dat$A == 1]^2), n_treated = sum(dat$A),
    ESS_control = sum(w[dat$A == 0])^2 / sum(w[dat$A == 0]^2), n_control = sum(1 - dat$A))
}

# ---- real data: loaders return list(name, X, A, Y, binary) ----
load_rhc <- function(path = "data/rhc.csv") {
  d <- read.csv(path, stringsAsFactors = TRUE)
  d[[1]] <- NULL                                              # first column is a row number
  A <- as.integer(d$swang1 == "RHC"); Y <- as.integer(d$dth30 == "Yes")
  drop <- c("swang1", "death", "dth30", "sadmdte", "dschdte", "dthdte", "lstctdte",   # same drop list as Karim (2026)
            "t3d30", "ptid", "cat2", "adld3p", "urin1")
  covs <- droplevels(d[, setdiff(names(d), drop), drop = FALSE])
  X <- as.data.frame(model.matrix(~ . - 1, data = covs))
  colnames(X) <- make.names(colnames(X), unique = TRUE)
  list(name = "RHC", X = X, A = A, Y = Y, binary = TRUE)
}
load_ihdp <- function(r = 1, dir = "data/ihdp") {
  d <- read.csv(file.path(dir, sprintf("ihdp_npci_%d.csv", r)), header = FALSE)
  names(d) <- c("treatment", "y_factual", "y_cfactual", "mu0", "mu1", paste0("x", 1:25))
  list(name = sprintf("IHDP #%d", r), X = d[, paste0("x", 1:25)], A = d$treatment, Y = d$y_factual,
       binary = FALSE, truth = mean(d$mu1 - d$mu0), mu0 = d$mu0, mu1 = d$mu1)
}
read_nsw <- function(file) {
  d <- read.table(file, header = FALSE)
  names(d) <- c("treat", "age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75", "re78")
  d
}
load_lalonde <- function(setting = c("psid", "experimental"), dir = "data/lalonde") {
  setting <- match.arg(setting)
  tr <- read_nsw(file.path(dir, "nswre74_treated.txt")); ct <- read_nsw(file.path(dir, "nswre74_control.txt"))
  benchmark <- mean(tr$re78) - mean(ct$re78)                   # randomized: the simple difference is the answer
  d <- if (setting == "experimental") rbind(tr, ct) else rbind(tr, read_nsw(file.path(dir, "psid_controls.txt")))
  covs <- c("age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75")
  list(name = paste0("LaLonde (", setting, ")"), X = d[, covs], A = d$treat, Y = d$re78, binary = FALSE, benchmark = benchmark)
}

# ---- learners: "glm" (main effects) or "rf" (random forest) ----
learn_g <- function(X, A, newX, learner = "glm", ntree = 100) {
  if (learner == "glm") {
    fit <- suppressWarnings(glm(A ~ ., data = data.frame(A = A, X), family = binomial))
    return(unname(suppressWarnings(predict(fit, newX, type = "response"))))
  }
  fit <- randomForest::randomForest(x = X, y = factor(A, levels = 0:1), ntree = ntree)
  unname(predict(fit, newX, type = "prob")[, "1"])
}
learn_Q <- function(X, A, Y, newX, binary, learner = "glm", ntree = 100) {
  tr <- data.frame(A = A, X); n1 <- data.frame(A = 1, newX); n0 <- data.frame(A = 0, newX)
  if (learner == "glm") {
    fit <- if (binary) suppressWarnings(glm(Y ~ ., data = data.frame(Y = Y, tr), family = binomial))
           else lm(Y ~ ., data = data.frame(Y = Y, tr))
    pr <- function(nd) unname(suppressWarnings(predict(fit, nd, type = "response")))
    return(list(Q1 = pr(n1), Q0 = pr(n0)))
  }
  if (binary) {
    fit <- randomForest::randomForest(x = tr, y = factor(Y, levels = 0:1), ntree = ntree)
    pr <- function(nd) unname(predict(fit, nd, type = "prob")[, "1"])
  } else {
    fit <- randomForest::randomForest(x = tr, y = Y, ntree = ntree)
    pr <- function(nd) unname(predict(fit, nd))
  }
  list(Q1 = pr(n1), Q0 = pr(n0))
}

# full: train on everyone, predict the same people.  crossfit: each prediction comes from the other K - 1 folds
nuisance <- function(d, learner = "glm", scheme = c("full", "crossfit"), K = 5, seed = 1,
                     bounds = c(0.025, 0.975), ntree = 100) {
  scheme <- match.arg(scheme)
  n <- length(d$A)
  set.seed(seed)
  fold <- if (scheme == "full") rep(1L, n) else sample(rep(seq_len(K), length.out = n))
  out <- data.frame(Q1 = numeric(n), Q0 = numeric(n), g_raw = numeric(n))
  for (k in unique(fold)) {
    te <- fold == k; tr <- if (scheme == "full") rep(TRUE, n) else !te
    q <- learn_Q(d$X[tr, , drop = FALSE], d$A[tr], d$Y[tr], d$X[te, , drop = FALSE], d$binary, learner, ntree)
    out$Q1[te] <- q$Q1; out$Q0[te] <- q$Q0
    out$g_raw[te] <- learn_g(d$X[tr, , drop = FALSE], d$A[tr], d$X[te, , drop = FALSE], learner, ntree)
  }
  if (d$binary) { out$Q1 <- pmin(pmax(out$Q1, 0), 1); out$Q0 <- pmin(pmax(out$Q0, 0), 1) }
  out$g <- pmin(pmax(out$g_raw, bounds[1]), bounds[2])
  out$fold <- fold
  out
}

panel <- function(d, nu, label = "") {                        # one set of nuisances -> four estimators
  dd <- list(A = d$A, Y = d$Y)
  a <- est_aipw(dd, nu); t <- est_tmle(dd, nu, ybounds = if (d$binary) c(0, 1) else range(d$Y))
  data.frame(scheme = label, estimator = c("g-computation", "IPTW (Hajek)", "AIPW", "TMLE"),
             est = c(est_plugin(nu), est_ipw(dd, nu), a[["est"]], t[["est"]]),
             se = c(NA, NA, a[["se"]], t[["se"]]), lo = c(NA, NA, a[["lo"]], t[["lo"]]), hi = c(NA, NA, a[["hi"]], t[["hi"]]))
}
residual_check <- function(d, nu) {                           # is the learner grading its own exam?
  QA <- ifelse(d$A == 1, nu$Q1, nu$Q0); H <- d$A / nu$g - (1 - d$A) / (1 - nu$g)
  c(resid_rms = sqrt(mean((d$Y - QA)^2)), correction = mean(H * (d$Y - QA)))
}

# ---- diagnostics: overlap and balance ----
smd <- function(x, A, w = rep(1, length(A))) {                # standardized mean difference
  m1 <- weighted.mean(x[A == 1], w[A == 1]); m0 <- weighted.mean(x[A == 0], w[A == 0])
  s <- sqrt((var(x[A == 1]) + var(x[A == 0])) / 2)
  if (s == 0) 0 else (m1 - m0) / s
}
balance_table <- function(X, A, g) {
  w <- ifelse(A == 1, 1 / g, 1 / (1 - g))
  data.frame(variable = names(X), before = sapply(X, smd, A = A), after_IPW = sapply(X, smd, A = A, w = w), row.names = NULL)
}
overlap_summary <- function(g_raw, A, bounds = c(0.025, 0.975)) {
  gb <- pmin(pmax(g_raw, bounds[1]), bounds[2]); w <- ifelse(A == 1, 1 / gb, 1 / (1 - gb))
  c(min_g = min(g_raw), max_g = max(g_raw), pct_clipped = 100 * mean(g_raw < bounds[1] | g_raw > bounds[2]),
    max_weight = max(w), ESS_treated = sum(w[A == 1])^2 / sum(w[A == 1]^2), ESS_control = sum(w[A == 0])^2 / sum(w[A == 0]^2))
}

# ---- plotting helpers (base graphics only) ----
COL <- list(red = "#D9480F", teal = "#0B7285", ink = "#1C1B19", muted = "#6B665E",
            gold = "#C98A0C", plum = "#7B3F8C", rule = "#DDD6CA")
lab_par <- function(...) par(mar = c(4.5, 5, 1.5, 1), las = 1, bty = "n", col.axis = COL$muted, ...)

overlap_plot <- function(g_raw, A, xlab = "estimated propensity score  g-hat(X)", labels = c("A = 1", "A = 0"), breaks = 40) {
  br <- seq(0, 1, length.out = breaks + 1)                    # mirrored histogram: treated up, control down
  h1 <- hist(pmin(pmax(g_raw[A == 1], 0), 1), breaks = br, plot = FALSE)$counts
  h0 <- hist(pmin(pmax(g_raw[A == 0], 0), 1), breaks = br, plot = FALSE)$counts
  lab_par()
  plot(NA, xlim = c(0, 1), ylim = c(-max(h0), max(h1)), axes = FALSE, xlab = xlab, ylab = "count")
  axis(1); yt <- pretty(c(-max(h0), max(h1))); axis(2, at = yt, labels = abs(yt))
  rect(br[-length(br)], 0, br[-1], h1, col = adjustcolor(COL$red, .8), border = "white")
  rect(br[-length(br)], 0, br[-1], -h0, col = adjustcolor(COL$teal, .8), border = "white")
  abline(h = 0)
  text(0.98, max(h1) * .85, labels[1], adj = 1, col = COL$red, font = 2)
  text(0.98, -max(h0) * .85, labels[2], adj = 1, col = COL$teal, font = 2)
}

forest_plot <- function(lab, est, lo = NA, hi = NA, ref = NULL, ref_lab = "", col = COL$ink, xlab = "estimate", xlim = NULL) {
  n <- length(est); y <- n:1
  rng <- range(c(est, lo, hi, ref), na.rm = TRUE)
  if (is.null(xlim)) xlim <- rng + c(-1, 1) * diff(rng) * 0.08
  op <- par(mar = c(4.5, min(22, max(8, 0.62 * max(nchar(lab)) + 1)), 1.5, 1), las = 1, bty = "n", col.axis = COL$muted)
  on.exit(par(op))
  plot(NA, xlim = xlim, ylim = c(0.5, n + 0.5), axes = FALSE, xlab = xlab, ylab = "")
  abline(h = y, col = "#F0EBE3")
  if (!is.null(ref)) {
    abline(v = ref, lty = 2, col = COL$muted); pos <- (ref[1] - xlim[1]) / diff(xlim)
    mtext(ref_lab, 3, at = ref[1], adj = if (pos > 0.7) 1 else if (pos < 0.3) 0 else 0.5, cex = .8, col = COL$muted)
  }
  ok <- !is.na(lo)
  segments(lo[ok], y[ok], hi[ok], y[ok], lwd = 3, col = rep_len(col, n)[ok])
  points(est, y, pch = 19, cex = 1.5, col = rep_len(col, n))
  axis(1); axis(2, at = y, labels = lab, tick = FALSE, col.axis = COL$ink)
}

| 格 | 在看什麼 | 投影片 |
|---|---|---|
| 01 | 隨機分派：直接相減就有答案 | 37–39 |
| 02–04 | 換成 PSID 對照：兩組差到不能比，propensity 幾乎分開 | 40、62 |
| 05–06 | 八個估計值各說各話；權重集中在少數人身上 | 52–53、88 |
| 07 | trimming：答案變了，因為題目變了 | 39 |

> 資料：LaLonde (1986) 的 NSW 職訓實驗，使用 Dehejia &amp; Wahba 整理的版本（users.nber.org/~rdehejia/data）：NSW 受訓者 185 人、NSW 隨機對照 260 人、PSID 對照 2,490 人。結果是 1978 年收入（美元），共變數：年齡、教育、黑人、西語裔、已婚、無學位、1974 與 1975 年收入。

## 01　隨機分派：基準答案

In [ ]:
ex <- load_lalonde("experimental")
c(trainees = sum(ex$A), randomized_controls = sum(1 - ex$A), naive_difference = ex$benchmark,
  regression_adjusted = unname(coef(lm(ex$Y ~ ex$A + as.matrix(ex$X)))[2]))

> **看什麼：**
>
> 隨機分派時，直接相減就是受訓者族群的效果：約 1,794 美元。加上共變數調整是 1,676，差不多——設計本身把兩組變得可以比。

## 02　換成 PSID 對照組

In [ ]:
d <- load_lalonde("psid")
grp <- data.frame(mean_trainees = sapply(d$X[d$A == 1, ], mean), mean_PSID = sapply(d$X[d$A == 0, ], mean), SMD = sapply(d$X, smd, A = d$A))
round(grp, 2)
c(trainees = sum(d$A), PSID_controls = sum(1 - d$A), naive_difference = mean(d$Y[d$A == 1]) - mean(d$Y[d$A == 0]))

> **看什麼：**
>
> 受訓者的 1974、1975 年收入只有 PSID 的零頭，年輕、沒結婚、沒學位的比例高很多——SMD 動輒超過 1。直接相減變成 −15,205 美元：不是訓練害的，是兩群人根本不是同一種人。

## 03　overlap：橋在哪裡？　<small>（投影片 40）</small>

In [ ]:
nu <- nuisance(d, "glm", "crossfit", seed = 1)
round(overlap_summary(nu$g_raw, d$A), 3)
overlap_plot(nu$g_raw, d$A, labels = c("NSW trainees (185)", "PSID controls (2,490)"))

> **看什麼：**
>
> 大部分 PSID 對照的 ĝ 擠在 0 附近，受訓者分散在右邊——兩座山幾乎沒有交集。約 80% 的人 ĝ &lt; 0.025，被截斷規則碰到。這不是估計器的問題，是資料裡**根本沒有**「長得像受訓者、卻沒受訓」的人可以比。

## 04　和 NHEFS 對照一下

| | NHEFS（Lab 5） | LaLonde × PSID |
|---|---|---|
| ĝ 範圍 | 0.05–0.78 | ≈ 0–0.96 |
| 被截斷的比例 | 0% | 約 80% |
| 治療組有效樣本數 / 人數 | 約 326 / 403 | 約 36 / 185 |

有效樣本數（ESS）掉到 36：權重加總之後，185 個受訓者只相當於 36 個人的資訊量——少數幾個人扛著整個答案。

## 05　八個估計值，各說各話　<small>（投影片 88）</small>

In [ ]:
allp <- do.call(rbind, lapply(c("glm", "rf"), function(lr) do.call(rbind, lapply(c("full", "crossfit"), function(sc)
  panel(d, nuisance(d, lr, sc, seed = 1, ntree = 100), paste(lr, sc))))))
allp[, c("est", "lo", "hi")] <- round(allp[, c("est", "lo", "hi")]); allp$se <- round(allp$se)
allp
forest_plot(paste(allp$scheme, allp$estimator), allp$est, allp$lo, allp$hi, ref = ex$benchmark,
            ref_lab = "experimental benchmark 1,794 (NSW population)", xlab = "difference in 1978 earnings (US$)",
            col = ifelse(grepl("IPTW", allp$estimator), COL$gold, ifelse(grepl("full", allp$scheme), COL$red, COL$teal)))

> **看什麼：**
>
> 從 −14,000（IPTW）到 +800，AIPW 和 TMLE 在同一組 nuisance 下還能差 3,700–4,400 美元（GLM cross-fit：AIPW −12、TMLE −4,422）。RF full sample 的 CI 只有 ±300，又是「又錯、又很有自信」。Karim 的 Table 9 一樣：−733 到 −14,629。這時候「方法之間一致」完全不成立——這本身就是一個警訊。投影片 88 說 AIPW 在 near-positivity violation 下不穩，在這裡看得最清楚。

<details><summary><b>想一想：那 1,794 是不是這裡的真值？誰離它最近誰就對？</b></summary>

不是。1,794 是 **NSW 受訓者族群**的效果（接近 ATT）。這裡估的是「受訓者 + PSID」合併族群的 ATE，那是另一個 estimand，大部分人是根本不會去參加職訓的一般人。1,794 只能當方向和量級的參考，不能拿來算 coverage（Karim 2026 也這樣提醒）。

</details>

## 06　權重集中在誰身上？　<small>（投影片 52–53）</small>

In [ ]:
w <- ifelse(d$A == 1, 1 / nu$g, 1 / (1 - nu$g))
wt <- w[d$A == 1]
c(max_trainee_weight = max(wt), n_at_cap = sum(wt >= 39.99),
  top10_share_of_weight = sum(sort(wt, decreasing = TRUE)[1:10]) / sum(wt), ESS = sum(wt)^2 / sum(wt^2))
par(mfrow = c(1, 2), mar = c(4.5, 4.5, 2.5, 1), las = 1, bty = "n")
hist(log10(wt), breaks = 30, col = adjustcolor(COL$red, .7), border = "white", main = "trainee weights 1 / g-hat",
     xaxt = "n", xlab = "weight (log scale; capped at 40)", ylab = "trainees")
axis(1, at = log10(c(1, 2, 5, 10, 20, 40)), labels = c(1, 2, 5, 10, 20, 40))
cs <- cumsum(sort(wt, decreasing = TRUE)) / sum(wt)            # cumulative share of total weight, heaviest first
plot(seq_along(cs), cs, type = "s", lwd = 3, col = COL$red, ylim = c(0, 1),
     xlab = "trainees, heaviest weight first", ylab = "share of total weight", main = "who carries the answer?")
abline(0, 1 / length(cs), lty = 2, col = COL$muted)
points(10, cs[10], pch = 19, cex = 1.6); text(10, cs[10], sprintf("  top 10 = %.0f%%", 100 * cs[10]), adj = 0)
legend("bottomright", c("actual", "if every trainee counted equally"), lty = c(1, 2), lwd = c(3, 1), col = c(COL$red, COL$muted), bty = "n")

> **看什麼：**
>
> - 左圖：大部分受訓者的權重在 1–1.5 之間——他們的 ĝ 接近 0.9，一看就「不像 PSID 的人」。但最右邊有 6 個人 ĝ 掉到 0.025 以下，權重被截在上限 40——他們長得太像 PSID 的一般人，模型幾乎不相信他們會受訓。
> - 右圖：**前 10 個人（185 人裡的 5%）扛了將近一半（約 49%）的總權重**，有效樣本數只剩 36。
> - 這 10 個人 1978 年的收入多 1,000 或少 1,000，答案就跟著跑。AIPW 的修正項是「權重 × 殘差」，所以同樣受影響（投影片 88：AIPW has the same weaknesses of IPW when it comes to positivity）。

## 07　Trimming：答案變了，因為題目變了　<small>（投影片 39）</small>

只保留 ĝ 在 [ε, 1 − ε] 的人，再做一次 cross-fit AIPW（= DML）。

In [ ]:
trim <- do.call(rbind, lapply(c(0, 0.05, 0.10), function(eps) {
  keep <- nu$g_raw >= eps & nu$g_raw <= 1 - eps
  dk <- list(X = d$X[keep, ], A = d$A[keep], Y = d$Y[keep], binary = FALSE)
  a <- est_aipw(dk, nuisance(dk, "glm", "crossfit", seed = 1))
  data.frame(rule = if (eps == 0) "keep all" else sprintf("keep g-hat in [%.2f, %.2f]", eps, 1 - eps),
             n_kept = sum(keep), trainees_kept = sum(dk$A), share_kept = round(mean(keep), 2),
             DML = round(a[["est"]]), lo = round(a[["lo"]]), hi = round(a[["hi"]]))
}))
trim

> **看什麼：**
>
> 修到 [0.05, 0.95] 只剩 420 人（16%），[0.10, 0.90] 剩 304 人（11%）——和 Karim 的 16%、11% 一樣。估計值跳到正的，CI 寬到 ±2,500–2,900。這不是「修好了原本的問題」：被丟掉的 84% 大部分是 PSID 的一般人，剩下的是「長得有點像受訓者」的子族群。estimand 從合併族群的 ATE 變成了 ψ<sub>ε</sub> = E[Q₁ − Q₀ | ε ≤ g ≤ 1−ε]。縮小題目可以是合理的選擇，但要明講。

### AIPW、TMLE、DML 像三條導航路線。跨河的橋不存在時，換導航 App 不會把橋生出來。

positivity 有兩層：**識別**（資料裡有沒有可比的人）和 **precision**（有的話夠不夠多）。Lab 4 第 06 格是第二層；這裡是第一層壞掉的樣子。處理方式是回到設計：換對照組、重新定義目標族群（例如改估 ATT）、或明講外推的假設——不是換一個更厲害的 learner。